In [ ]:
import os
import re
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
from tqdm import tqdm

MERGED_DIR = Path(r"샘플데이터/03.합성데이터/1.모터_감속기")
OUTPUT_DIR = Path(r"샘플데이터/03.합성데이터/1.모터_감속기_시계열")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

THRESHOLD_PERCENTILE = 90
MIN_BAND_HEIGHT = 2

In [4]:
# 합성 이미지 파일명 파싱
# 형식: YY_MM_DD_hh_mm_ss_nn_zsplitNNN_merged_SensorType.png
MERGED_PATTERN = re.compile(
    r"^(\d{2}_\d{2}_\d{2}_\d{2}_\d{2}_\d{2})"
    r"_(\d{2})"
    r"_(zsplit\d{3})"
    r"_merged_(.+)\.png$"
)

def parse_merged_filename(fname):
    m = MERGED_PATTERN.match(fname)
    if not m:
        return None
    ts, seq, zsplit, sensor = m.groups()
    return {"timestamp": ts, "seq": seq, "zsplit": zsplit, "sensor": sensor}


def parse_path_metadata(png_path):
    """경로에서 vehicle/fault_class/date 추출.
    구조: MERGED_DIR / vehicle / fault_class / date / sensor / 파일명
    """
    rel = png_path.relative_to(MERGED_DIR)
    parts = rel.parts  # (vehicle, fault_class, date, sensor, filename)
    if len(parts) < 5:
        return None
    vehicle, fault_class, date, sensor_dir = parts[0], parts[1], parts[2], parts[3]
    info = parse_merged_filename(parts[4])
    if info is None:
        return None
    info["vehicle"] = vehicle
    info["fault_class"] = fault_class
    info["date"] = date
    info["group_key"] = f"{info['timestamp']}_{info['seq']}_{info['zsplit']}_{info['sensor']}"
    return info


# 전체 합성 PNG 수집
all_pngs = sorted(MERGED_DIR.rglob("*_merged_*.png"))
print(f"발견된 합성 PNG: {len(all_pngs)}개")

발견된 합성 PNG: 81개


In [5]:
# 시계열 특징 추출 함수 (모터분석.ipynb와 동일)

def image_to_energy_map(image):
    arr = np.asarray(image.convert("RGB"), dtype=np.float32) / 255.0
    energy = np.sqrt(np.mean(arr ** 2, axis=2))
    energy = (energy - energy.min()) / (energy.max() - energy.min() + 1e-8)
    return energy


def extract_timeseries_from_spectrogram(
    image,
    threshold_percentile=90,
    min_band_height=2,
    freq_from_bottom=True,
):
    energy = image_to_energy_map(image)
    height, width = energy.shape
    threshold = np.percentile(energy, threshold_percentile)

    rows = []
    for x in range(width):
        column = energy[:, x]
        active_y = np.where(column >= threshold)[0]
        rms = float(np.sqrt(np.mean(column ** 2)))

        if len(active_y) >= min_band_height:
            peak_y = int(np.argmax(column))
            if freq_from_bottom:
                active_bins = height - 1 - active_y
                peak_freq_bin = height - 1 - peak_y
            else:
                active_bins = active_y
                peak_freq_bin = peak_y
            band_start = int(active_bins.min())
            band_end = int(active_bins.max())
        else:
            peak_freq_bin = np.nan
            band_start = np.nan
            band_end = np.nan

        rows.append({
            "time_idx": x,
            "peak_freq_bin": peak_freq_bin,
            "band_start": band_start,
            "band_end": band_end,
            "rms": rms,
        })

    return pd.DataFrame(rows)

In [6]:
# 모든 합성 PNG → 시계열 CSV 변환
skipped = []
done = 0

for png_path in tqdm(all_pngs, desc="변환 중"):
    info = parse_path_metadata(png_path)
    if info is None:
        skipped.append(str(png_path))
        continue

    # 출력 경로: 원본 디렉터리 구조 유지
    rel_dir = png_path.relative_to(MERGED_DIR).parent  # vehicle/fault_class/date/sensor
    out_dir = OUTPUT_DIR / rel_dir
    out_dir.mkdir(parents=True, exist_ok=True)
    csv_path = out_dir / f"{info['group_key']}_timeseries.csv"

    if csv_path.exists():
        done += 1
        continue  # 이미 변환된 파일 건너뜀

    image = Image.open(png_path)
    ts_df = extract_timeseries_from_spectrogram(
        image,
        threshold_percentile=THRESHOLD_PERCENTILE,
        min_band_height=MIN_BAND_HEIGHT,
        freq_from_bottom=True,
    )

    # 메타데이터 앞에 삽입
    for col, val in [
        ("zsplit",      info["zsplit"]),
        ("timestamp",   info["timestamp"]),
        ("sensor",      info["sensor"]),
        ("date",        info["date"]),
        ("fault_class", info["fault_class"]),
        ("vehicle",     info["vehicle"]),
        ("group_key",   info["group_key"]),
    ]:
        ts_df.insert(0, col, val)

    ts_df.to_csv(csv_path, index=False, encoding="utf-8-sig")
    done += 1

print(f"\n완료: {done}개 처리 / {len(all_pngs)}개 전체")
if skipped:
    print(f"파싱 실패 (건너뜀): {len(skipped)}개")
    for s in skipped:
        print(f"  {s}")
print(f"저장 위치: {OUTPUT_DIR}")

ImportError: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html

In [ ]:
# 결과 요약
csv_files = sorted(OUTPUT_DIR.rglob("*.csv"))
print(f"생성된 CSV 총: {len(csv_files)}개")

# fault_class별 수량
from collections import Counter
fault_counts = Counter()
for f in csv_files:
    parts = f.relative_to(OUTPUT_DIR).parts
    if len(parts) >= 2:
        fault_counts[parts[1]] += 1

print("\nfault_class별 CSV 수:")
for k, v in sorted(fault_counts.items()):
    print(f"  {k}: {v}개")

# 샘플 1개 미리보기
if csv_files:
    sample_csv = csv_files[0]
    print(f"\n샘플 CSV: {sample_csv.name}")
    pd.read_csv(sample_csv, encoding="utf-8-sig").head()